# 🐦 Detecção de Pássaros com YOLOv8

**Disciplina:** Visão Computacional  
**Professor:** Saulo Santos  
**Instituição:** UNIARA / UNICEP  

---

## Objetivo

Neste projeto, vamos treinar um modelo **YOLOv8** para detectar e identificar espécies de pássaros em imagens.  
O modelo será capaz de localizar os pássaros com uma *bounding box* e classificar a espécie.

### Espécies detectadas:
| ID | Espécie |
|----|---------|
| 0  | Pardal  |
| 1  | Pomba   |
| 2  | Tucano  |
| 3  | Papagaio|
| 4  | Beija-flor |

---

### Estrutura do Notebook
1. Configuração do ambiente
2. Verificação do dataset
3. Treinamento do modelo
4. Análise dos gráficos de desempenho
5. Matriz de confusão
6. Inferência em imagens de teste
7. Exportação do modelo final


## 1. Configuração do Ambiente 🔧

Importamos as bibliotecas e configuramos as pastas de saída dos resultados.

In [ ]:
from ultralytics import YOLO, settings
import os
import torch
import yaml
from pathlib import Path
from IPython.display import Image, display
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Salvar resultados localmente (não em /root/.config)
settings.update({'runs_dir': './runs'})

# Verificar disponibilidade de GPU
cuda_ok = torch.cuda.is_available()
device_info = torch.cuda.get_device_name(0) if cuda_ok else 'CPU (sem GPU NVIDIA)'
print(f"🖥️  Dispositivo: {device_info}")
print(f"⚡ CUDA disponível: {cuda_ok}")

if not cuda_ok:
    print("\n💡 DICA: Sem GPU, o treinamento será mais lento.")
    print("   Para CPU use epochs=5 e imgsz=416 para testes rápidos.")

## 2. Verificação do Dataset 📂

Antes de treinar, vamos garantir que o dataset está na estrutura correta.

### Estrutura esperada:
```
dataset_passaros/
├── data.yaml
├── train/
│   ├── images/   ← ~70% das imagens
│   └── labels/   ← arquivos .txt com anotações
├── valid/
│   ├── images/   ← ~20% das imagens
│   └── labels/
└── test/
    ├── images/   ← ~10% das imagens
    └── labels/
```

> **Obtendo o dataset:**  
> Acesse [Roboflow Universe](https://universe.roboflow.com/) e pesquise por **"bird detection yolov8"**.  
> Baixe no formato **YOLOv8** e extraia dentro de `dataset_passaros/`.

In [ ]:
# ── Caminho para o data.yaml ──────────────────────────────────────────────────
# ⚠️ EDITE AQUI se o nome da pasta for diferente após extrair o zip do Roboflow
DATASET_YAML = '../dataset_passaros/data.yaml'

# ─────────────────────────────────────────────────────────────────────────────

dataset_path = Path(DATASET_YAML)

if not dataset_path.exists():
    print(f"❌ Arquivo '{dataset_path}' não encontrado!")
    print("\n📌 Execute o script preparar_dataset.py para instruções:")
    print("   poetry run python preparar_dataset.py")
else:
    print(f"✅ data.yaml encontrado: {dataset_path.resolve()}")
    
    # Ler e exibir conteúdo do data.yaml
    with open(dataset_path) as f:
        data_cfg = yaml.safe_load(f)
    
    print(f"\n🐦 Classes no dataset: {data_cfg.get('nc', '?')}")
    for idx, nome in data_cfg.get('names', {}).items():
        print(f"   [{idx}] {nome}")
    
    # Contar imagens
    base = dataset_path.parent
    for split in ['train', 'valid', 'test']:
        img_dir = base / split / 'images'
        if img_dir.exists():
            count = len(list(img_dir.glob('*.jpg'))) + len(list(img_dir.glob('*.png')))
            print(f"   📁 {split:6s}: {count} imagens")

## 3. Treinamento do Modelo 🔥

Vamos usar **Transfer Learning** com o modelo `yolov8n.pt` (Nano) como base.  
O modelo já aprendeu a detectar formas e texturas gerais — nós apenas o "ensinamos" a reconhecer pássaros.

### Parâmetros principais:
| Parâmetro | Valor | Descrição |
|-----------|-------|-----------|
| `data`    | data.yaml | Caminho do dataset |
| `epochs`  | 50    | Rodadas de estudo (use 5 para testes rápidos) |
| `imgsz`   | 640   | Tamanho das imagens (padrão YOLO) |
| `batch`   | 8     | Imagens processadas por vez (reduza se faltar memória) |
| `model`   | yolov8n.pt | Modelo base pré-treinado (Nano = rápido) |

In [ ]:
# ── CONFIGURAÇÕES DE TREINAMENTO ──────────────────────────────────────────────
EPOCHS  = 50    # Aumente para 100+ para melhor resultado final
IMGSZ   = 640   # Tamanho padrão YOLOv8
BATCH   = 8     # Reduza para 4 ou 2 se der erro de memória
MODEL   = 'yolov8n.pt'  # Nano (mais rápido). Alternativa: yolov8s.pt (Small)
# ─────────────────────────────────────────────────────────────────────────────

print(f"🚀 Iniciando treinamento...")
print(f"   Modelo base : {MODEL}")
print(f"   Épocas      : {EPOCHS}")
print(f"   Img size    : {IMGSZ}")
print(f"   Batch size  : {BATCH}")
print(f"   Dataset     : {DATASET_YAML}")
print("\n" + "─" * 50)

# Carregar modelo pré-treinado (baixa automaticamente na primeira vez)
model = YOLO(MODEL)

# ── TREINAMENTO ───────────────────────────────────────────────────────────────
results = model.train(
    data=DATASET_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    name='passaros_yolov8n',   # nome da pasta de resultados em runs/detect/
    patience=10,               # para cedo se não houver melhora por 10 épocas
    save=True,
    plots=True,
    verbose=True,
)

print(f"\n✅ Treinamento concluído!")
print(f"📁 Resultados salvos em: {results.save_dir}")

## 4. Análise dos Gráficos de Desempenho 📊

O YOLO gera automaticamente o arquivo `results.png` com as curvas de aprendizado.

### O que observar:
- **Loss (Perda):** Deve **diminuir** ao longo das épocas → o modelo está aprendendo.
- **mAP50:** Deve **aumentar** ao longo das épocas → o modelo está ficando mais preciso.
- **Overfitting:** Se `train_loss` cai mas `val_loss` sobe, o modelo está "decorando" as imagens.

In [ ]:
save_dir = Path(results.save_dir)
print(f"📁 Pasta de resultados: {save_dir}")

# Exibir gráfico de resultados
results_img = save_dir / 'results.png'
if results_img.exists():
    print("\n📈 Gráficos de treinamento:")
    display(Image(filename=str(results_img), width=900))
else:
    print("⚠️ Arquivo results.png não encontrado ainda.")

### 📖 Como interpretar o gráfico `results.png`

**Curvas de Perda (Loss) — quanto menor, melhor 📉**

| Curva | O que mede | Comportamento ideal |
|-------|-----------|--------------------|
| `train/box_loss` | Erro na posição da bounding box | Cai progressivamente |
| `train/cls_loss` | Erro na classificação da espécie | Cai progressivamente |
| `val/box_loss` | Erro de localização na validação | Cai junto com treino |
| `val/cls_loss` | Erro de classe na validação | Cai junto com treino |

**Métricas de Desempenho — quanto maior, melhor 📈**

| Métrica | O que mede |
|---------|----------|
| `metrics/precision` | De tudo que o modelo disse ser pássaro, quantos eram? |
| `metrics/recall` | De todos os pássaros reais, quantos o modelo encontrou? |
| `metrics/mAP50` | Nota geral (aceitando 50% de sobreposição da caixa) |
| `metrics/mAP50-95` | Nota rigorosa (média de 50% a 95% de sobreposição) |

## 5. Matriz de Confusão 😵

A Matriz de Confusão mostra onde o modelo acerta e onde confunde espécies.

- **Diagonal principal** = acertos (era Tucano → disse Tucano ✅)
- **Fora da diagonal** = erros (era Pardal → disse Pomba ❌)

In [ ]:
confusion_img = save_dir / 'confusion_matrix.png'
if confusion_img.exists():
    print("🔲 Matriz de Confusão:")
    display(Image(filename=str(confusion_img), width=700))
else:
    print("⚠️ Arquivo confusion_matrix.png não encontrado.")

## 6. Inferência em Imagens de Validação 🔍

Vamos visualizar as previsões do modelo em imagens que ele **nunca viu** durante o treino.

In [ ]:
# Exibir o lote de validação com as previsões desenhadas
val_pred = save_dir / 'val_batch0_pred.jpg'
if val_pred.exists():
    print("🖼️  Previsões no conjunto de validação (val_batch0_pred.jpg):")
    display(Image(filename=str(val_pred), width=900))
else:
    print("⚠️ Arquivo val_batch0_pred.jpg não encontrado.")

In [ ]:
# Inferência em uma imagem específica do conjunto de teste
# ── Edite o caminho abaixo para testar uma imagem sua ──────────────────────────
test_images_dir = Path('../dataset_passaros/test/images')

# Pegar a primeira imagem de teste disponível
test_imgs = list(test_images_dir.glob('*.jpg')) + list(test_images_dir.glob('*.png'))

if test_imgs:
    img_teste = str(test_imgs[0])
    print(f"🔍 Realizando inferência em: {img_teste}")
    
    # Carregar o melhor modelo treinado
    best_model = YOLO(str(save_dir / 'weights' / 'best.pt'))
    
    # Executar predição
    pred_results = best_model.predict(
        source=img_teste,
        conf=0.25,        # confiança mínima de 25%
        save=True,
        save_dir='./resultados_predicao',
        line_width=2,
    )
    
    # Mostrar resultado
    for r in pred_results:
        print(f"\n🐦 Detecções encontradas: {len(r.boxes)}")
        for box in r.boxes:
            cls_id  = int(box.cls[0])
            conf    = float(box.conf[0])
            nome_classe = best_model.names[cls_id]
            print(f"   → {nome_classe:12s} | Confiança: {conf:.1%}")
else:
    print("⚠️ Nenhuma imagem de teste encontrada em:", test_images_dir)
    print("   Certifique-se de que o dataset foi extraído corretamente.")

## 7. Validação Completa no Dataset de Teste 🎓

Rodamos a validação formal no conjunto de **teste** — as imagens que ficaram "trancadas no cofre"  
durante todo o treinamento. Esta é a nota real do modelo.

In [ ]:
print("📊 Avaliação final no conjunto de TESTE...")

best_model = YOLO(str(save_dir / 'weights' / 'best.pt'))

val_metrics = best_model.val(
    data=DATASET_YAML,
    split='test',
    verbose=True,
)

print("\n" + "═" * 50)
print("  RESULTADOS FINAIS NO CONJUNTO DE TESTE")
print("═" * 50)
print(f"  mAP@50       : {val_metrics.box.map50:.4f}")
print(f"  mAP@50-95    : {val_metrics.box.map:.4f}")
print(f"  Precision    : {val_metrics.box.mp:.4f}")
print(f"  Recall       : {val_metrics.box.mr:.4f}")
print("═" * 50)

## 8. Exportação do Modelo Final 💾

Salvamos o modelo treinado para uso futuro.  
O arquivo `best.pt` pode ser carregado em qualquer script Python com:
```python
model = YOLO('caminho/para/best.pt')
```

In [ ]:
import shutil

# Copiar o melhor modelo para uma pasta de fácil acesso
best_weights = save_dir / 'weights' / 'best.pt'
dest = Path('./modelo_final_passaros.pt')

if best_weights.exists():
    shutil.copy(best_weights, dest)
    tamanho_mb = dest.stat().st_size / (1024 * 1024)
    print(f"✅ Modelo exportado: {dest.resolve()}")
    print(f"   Tamanho: {tamanho_mb:.1f} MB")
    print(f"\n💡 Para usar o modelo:")
    print(f"   from ultralytics import YOLO")
    print(f"   model = YOLO('modelo_final_passaros.pt')")
    print(f"   results = model.predict(source='sua_imagem.jpg', conf=0.25)")
else:
    print("⚠️ Arquivo best.pt não encontrado. Execute o treinamento primeiro.")

## 9. Detecção em Vídeo MP4 com Círculos 🎥

Assim como o exemplo do professor com peixes, aqui vamos processar um vídeo MP4 frame a frame,
detectando pássaros e **desenhando um círculo colorido** ao redor de cada um.

> **💡 Como obter um vídeo de teste:**
> Baixe um vídeo do YouTube com pássaros (ex: *"birds in garden 4k"*) usando um site como
> [yt-dlp](https://github.com/yt-dlp/yt-dlp) e salve como `video_passaros.mp4` na raiz do projeto.

In [ ]:
import cv2
import math
from pathlib import Path
from ultralytics import YOLO
from IPython.display import Video, display

# ── CONFIGURAÇÕES ────────────────────────────────────────────────────────────
VIDEO_ENTRADA = '../video_passaros.mp4'    # ⚠️ coloque seu vídeo aqui
VIDEO_SAIDA   = '../passaros_detectados.mp4'
CONF_VIDEO    = 0.35    # confiança mínima (35%)

# Cores por espécie (formato BGR do OpenCV)
CORES = {
    'pardal':     (0,   200, 255),   # laranja
    'pomba':      (255, 180,   0),   # azul
    'tucano':     (0,   255, 100),   # verde
    'papagaio':   (0,    80, 255),   # vermelho
    'beija-flor': (255,   0, 200),   # roxo
}
COR_PADRAO = (0, 255, 255)  # amarelo
# ─────────────────────────────────────────────────────────────────────────────

modelo_path = str(save_dir / 'weights' / 'best.pt')
video_in = Path(VIDEO_ENTRADA)

if not video_in.exists():
    print(f'⚠️ Vídeo não encontrado: {video_in}')
    print('   Coloque um arquivo video_passaros.mp4 na raiz do projeto.')
else:
    print(f'📹 Processando vídeo: {video_in}')
    
    model_vid = YOLO(modelo_path)
    cap = cv2.VideoCapture(str(video_in))
    
    fps    = int(cap.get(cv2.CAP_PROP_FPS)) or 30
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(VIDEO_SAIDA, fourcc, fps, (width, height))
    
    frame_num = 0
    total_det = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_num += 1
        results = model_vid(frame, conf=CONF_VIDEO, verbose=False)
        
        det_frame = 0
        for result in results:
            for box in result.boxes:
                cls_id    = int(box.cls[0])
                conf_val  = float(box.conf[0])
                nome      = model_vid.names[cls_id]
                cor       = CORES.get(nome.lower(), COR_PADRAO)
                
                # Coordenadas da bbox
                x1, y1, x2, y2 = (int(box.xyxy[0][0]), int(box.xyxy[0][1]),
                                   int(box.xyxy[0][2]), int(box.xyxy[0][3]))
                
                # Centro e raio do círculo (cobre toda a bbox)
                cx = (x1 + x2) // 2
                cy = (y1 + y2) // 2
                raio = int(math.sqrt((x2-x1)**2 + (y2-y1)**2) / 2)
                
                # Desenhar círculo
                cv2.circle(frame, (cx, cy), raio, cor, 3)
                cv2.circle(frame, (cx, cy), 5, cor, -1)  # ponto central
                
                # Label com nome e confiança
                label = f'{nome} {conf_val:.0%}'
                (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.65, 2)
                tx, ty = cx - tw//2, cy - raio - 8
                cv2.rectangle(frame, (tx-4, ty-th-4), (tx+tw+4, ty+4), (0,0,0), -1)
                cv2.putText(frame, label, (tx, ty),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.65, cor, 2, cv2.LINE_AA)
                
                det_frame += 1
                total_det += 1
        
        # Contador no frame
        cv2.putText(frame, f'Passaros: {det_frame} | Frame: {frame_num}/{total}',
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
        
        writer.write(frame)
        
        if frame_num % 30 == 0:
            print(f'   {frame_num/total*100:.1f}% — Frame {frame_num}/{total} | Detecções: {total_det}')
    
    cap.release()
    writer.release()
    
    print(f'\n✅ Vídeo salvo: {VIDEO_SAIDA}')
    print(f'🐦 Total de pássaros detectados: {total_det}')
    
    # Exibir vídeo no notebook
    display(Video(VIDEO_SAIDA, width=700, embed=True))

### 🎨 Cores por espécie no vídeo

| Espécie | Cor do círculo |
|---------|---------------|
| Pardal | 🟠 Laranja |
| Pomba | 🔵 Azul |
| Tucano | 🟢 Verde |
| Papagaio | 🔴 Vermelho |
| Beija-flor | 🟣 Roxo |

> Para usar o script standalone (sem abrir o notebook), rode no terminal:
> ```bash
> poetry run python detectar_video_passaros.py --source video_passaros.mp4
> ```

## 10. Inferência via Terminal (Alternativa) 🖥️

Você também pode treinar e fazer inferência diretamente pelo terminal, sem abrir o Jupyter:

```bash
# Treinamento
poetry run yolo detect train \
    data=dataset_passaros/data.yaml \
    model=yolov8n.pt \
    epochs=50 \
    imgsz=640 \
    name=passaros_yolov8n

# Inferência em uma imagem
poetry run yolo detect predict \
    model=runs/detect/passaros_yolov8n/weights/best.pt \
    source=sua_imagem.jpg \
    conf=0.25
```

---

## 🎉 Conclusão

Você treinou com sucesso um detector de pássaros com YOLOv8!  
O modelo aprendeu a:
1. **Localizar** pássaros em imagens (bounding box)
2. **Classificar** a espécie detectada
3. **Quantificar** a confiança de cada detecção

Para melhorar os resultados, tente:
- Aumentar o número de épocas (`epochs=100`)
- Usar um modelo maior (`yolov8s.pt` ou `yolov8m.pt`)
- Adicionar mais imagens ao dataset
- Usar técnicas de *data augmentation*